# Claude API — Bedrock Gateway Notebook

This notebook reads credentials and endpoints from `~/.claude/settings.json`
and provides a reusable helper module for calling Claude via the
Salesforce Bedrock gateway.

## Structure
1. **`read_claude_settings()`** — reads and parses `~/.claude/settings.json`
2. **`get_claude_env()`** — extracts individual env vars with fallback to OS env
3. **`ClaudeClient`** — thin wrapper around `anthropic.AnthropicBedrock`
4. **Usage examples** — chat, streaming, and multi-turn conversation

## 1. Settings Reader Helper
Reads `~/.claude/settings.json`, extracts the `env` block, and exposes
individual keys. All downstream code calls `get_claude_env()` — never
hard-codes a URL or token.

In [ ]:
import json          # parse the JSON settings file
import os            # fallback to OS environment variables
import pathlib       # cross-platform path handling (~/.claude/settings.json)
from typing import Any, Dict, Optional

# ── Path to the Claude Code settings file ────────────────────────────────
SETTINGS_PATH = pathlib.Path.home() / '.claude' / 'settings.json'


def read_claude_settings(path: pathlib.Path = SETTINGS_PATH) -> Dict[str, Any]:
    """
    Read and parse ~/.claude/settings.json.

    Args:
        path : path to the settings file (default: ~/.claude/settings.json)

    Returns:
        The full parsed settings dict, e.g.:
        {
          'env': {'ANTHROPIC_BEDROCK_BASE_URL': '...', ...},
          'model': 'global.anthropic.claude-sonnet-4-6',
          'modelOverrides': {...},
          ...
        }

    Raises:
        FileNotFoundError : if settings.json does not exist
        json.JSONDecodeError: if the file is not valid JSON
    """
    if not path.exists():
        raise FileNotFoundError(
            f'Claude settings not found at {path}.\n'
            f'Install Claude Code or create the file manually.'
        )

    raw_text = path.read_text(encoding='utf-8')  # read as UTF-8 string
    settings = json.loads(raw_text)              # parse JSON → Python dict
    return settings


def get_claude_env(
    key: str,
    settings: Optional[Dict[str, Any]] = None,
    fallback_to_os: bool = True,
) -> Optional[str]:
    """
    Get a single environment variable from the Claude settings.

    Lookup order:
      1. settings['env'][key]  (from ~/.claude/settings.json)
      2. os.environ[key]       (if fallback_to_os=True)
      3. None

    Args:
        key            : env var name, e.g. 'ANTHROPIC_BEDROCK_BASE_URL'
        settings       : pre-loaded settings dict (loaded once and reused)
        fallback_to_os : if True, fall back to OS environment on miss

    Returns:
        The string value, or None if not found anywhere.
    """
    # Load settings lazily if not provided
    if settings is None:
        settings = read_claude_settings()

    # Extract the 'env' block from settings (may not exist)
    env_block = settings.get('env', {})   # default to empty dict if missing

    # Check settings first, then OS env, then return None
    value = env_block.get(key)            # returns None if key absent
    if value is None and fallback_to_os:
        value = os.environ.get(key)       # check OS environment as backup

    return value


def get_all_claude_env(settings: Optional[Dict[str, Any]] = None) -> Dict[str, str]:
    """
    Return the entire env block from settings as a dict.
    Merges OS environment on top of settings (settings take priority).

    Returns:
        dict of all env vars defined in ~/.claude/settings.json
    """
    if settings is None:
        settings = read_claude_settings()
    return dict(settings.get('env', {}))  # return a copy so caller can mutate


# ── Quick smoke test ──────────────────────────────────────────────────────
settings = read_claude_settings()         # load once; reuse below

bedrock_url  = get_claude_env('ANTHROPIC_BEDROCK_BASE_URL', settings)
base_url     = get_claude_env('ANTHROPIC_BASE_URL',         settings)
auth_token   = get_claude_env('ANTHROPIC_AUTH_TOKEN',       settings)
use_bedrock  = get_claude_env('CLAUDE_CODE_USE_BEDROCK',    settings)
skip_auth    = get_claude_env('CLAUDE_CODE_SKIP_BEDROCK_AUTH', settings)
ca_bundle    = get_claude_env('NODE_EXTRA_CA_CERTS',        settings)

print('=== Claude Settings Loaded ===')
print(f'ANTHROPIC_BEDROCK_BASE_URL : {bedrock_url}')
print(f'ANTHROPIC_BASE_URL         : {base_url}')
print(f'ANTHROPIC_AUTH_TOKEN       : {auth_token[:12]}...{auth_token[-4:] if auth_token else ""}')
print(f'CLAUDE_CODE_USE_BEDROCK    : {use_bedrock}')
print(f'CLAUDE_CODE_SKIP_BEDROCK_AUTH: {skip_auth}')
print(f'CA_BUNDLE                  : {ca_bundle}')
print(f'\nModel from settings        : {settings.get("model")}')
print(f'Model overrides            : {list(settings.get("modelOverrides",{}).keys())}')

## 2. ClaudeClient — Reusable API Client

Wraps `anthropic.AnthropicBedrock` with our settings helper so any
notebook or script can do:
```python
from claude_1 import ClaudeClient
client = ClaudeClient()
reply  = client.chat('What is attention?')
```

In [ ]:
import anthropic     # pip install anthropic
import ssl           # needed to set custom CA bundle for corporate proxy


class ClaudeClient:
    """
    Thin wrapper around anthropic.AnthropicBedrock that reads all
    configuration from ~/.claude/settings.json automatically.

    Usage:
        client = ClaudeClient()
        reply  = client.chat('Explain attention in one sentence.')
        print(reply)
    """

    def __init__(
        self,
        model: Optional[str] = None,
        settings_path: pathlib.Path = SETTINGS_PATH,
    ):
        """
        Args:
            model         : override the model ID (default: read from settings)
            settings_path : path to ~/.claude/settings.json
        """
        # Load settings once at construction time
        self._settings = read_claude_settings(settings_path)

        # ── Read all required values from settings ─────────────────────
        self.bedrock_url = get_claude_env('ANTHROPIC_BEDROCK_BASE_URL', self._settings)
        self.base_url    = get_claude_env('ANTHROPIC_BASE_URL',         self._settings)
        self.auth_token  = get_claude_env('ANTHROPIC_AUTH_TOKEN',       self._settings)
        self.ca_bundle   = get_claude_env('NODE_EXTRA_CA_CERTS',        self._settings)

        # Model: argument → settings.model → default fallback
        self.model = (
            model
            or self._settings.get('model')
            or 'global.anthropic.claude-sonnet-4-6'
        )

        # ── Build the httpx client kwargs for custom CA + proxy ─────────
        # The Salesforce gateway requires the corporate CA bundle for TLS
        http_client_kwargs = {}
        if self.ca_bundle and pathlib.Path(self.ca_bundle).exists():
            import httpx
            # Tell httpx to trust the corporate CA bundle
            http_client_kwargs['verify'] = self.ca_bundle

        # ── Instantiate the Anthropic Bedrock client ────────────────────
        # base_url points to the Salesforce Bedrock proxy
        # api_key carries the ANTHROPIC_AUTH_TOKEN as the bearer credential
        import httpx
        self._client = anthropic.Anthropic(
            base_url   = self.base_url,           # Salesforce proxy base URL
            api_key    = self.auth_token,          # auth token as API key
            http_client= httpx.Client(
                verify  = self.ca_bundle or True,  # corporate CA bundle
                headers = {'x-skip-bedrock-auth': '1'},  # skip AWS SigV4
            ),
        )

        print(f'ClaudeClient ready')
        print(f'  base_url : {self.base_url}')
        print(f'  model    : {self.model}')

    # ── Core chat method ───────────────────────────────────────────────────
    def chat(
        self,
        prompt: str,
        system: Optional[str] = None,
        max_tokens: int = 1024,
        temperature: float = 1.0,
    ) -> str:
        """
        Send a single user message and return the text response.

        Args:
            prompt      : user message text
            system      : optional system prompt
            max_tokens  : maximum tokens to generate
            temperature : sampling temperature (0=deterministic, 1=default)

        Returns:
            The assistant's reply as a plain string.
        """
        # Build the messages list with the user turn
        messages = [{'role': 'user', 'content': prompt}]

        # Assemble kwargs; only add system if provided
        kwargs = {
            'model'      : self.model,
            'max_tokens' : max_tokens,
            'messages'   : messages,
        }
        if system:
            kwargs['system'] = system       # top-level system prompt

        # Call the API and extract the text from the first content block
        response = self._client.messages.create(**kwargs)
        return response.content[0].text

    # ── Multi-turn conversation ────────────────────────────────────────────
    def converse(
        self,
        messages: list,
        system: Optional[str] = None,
        max_tokens: int = 1024,
    ) -> str:
        """
        Send a full conversation history and return the next assistant turn.

        Args:
            messages   : list of {'role': 'user'|'assistant', 'content': str}
            system     : optional system prompt
            max_tokens : max tokens to generate

        Returns:
            Assistant reply string.
        """
        kwargs = {
            'model'      : self.model,
            'max_tokens' : max_tokens,
            'messages'   : messages,
        }
        if system:
            kwargs['system'] = system

        response = self._client.messages.create(**kwargs)
        return response.content[0].text

    # ── Streaming chat ────────────────────────────────────────────────────
    def stream(
        self,
        prompt: str,
        system: Optional[str] = None,
        max_tokens: int = 1024,
    ) -> str:
        """
        Stream the response and print tokens as they arrive.
        Returns the fully assembled response string.
        """
        messages = [{'role': 'user', 'content': prompt}]
        kwargs   = {'model': self.model, 'max_tokens': max_tokens, 'messages': messages}
        if system:
            kwargs['system'] = system

        full_text = ''
        with self._client.messages.stream(**kwargs) as stream:
            for chunk in stream.text_stream:     # yields text deltas
                print(chunk, end='', flush=True) # print token as it arrives
                full_text += chunk
        print()   # newline after streaming finishes
        return full_text

    def __repr__(self) -> str:
        return f'ClaudeClient(model={self.model!r}, base_url={self.base_url!r})'

## 3. Instantiate the Client

In [ ]:
# Create a client — reads all config from ~/.claude/settings.json automatically
client = ClaudeClient()
print(client)

## 4. Simple Chat

In [ ]:
# ── Single-turn chat ──────────────────────────────────────────────────────
reply = client.chat('What is multi-head attention in one paragraph?')
print(reply)

In [ ]:
# ── Chat with a system prompt ─────────────────────────────────────────────
# System prompt sets the persona / context for the assistant
reply = client.chat(
    prompt = 'Explain gradient descent.',
    system = 'You are a concise ML tutor. Answer in 3 bullet points max.',
    max_tokens = 512,
)
print(reply)

## 5. Multi-Turn Conversation

In [ ]:
# Build a conversation history manually:
# Each turn is {'role': 'user' | 'assistant', 'content': <text>}
# The model sees the full history on every call.

conversation = []   # start with empty history

def chat_turn(user_msg: str) -> str:
    """Add a user turn, call the API, append assistant reply, return it."""
    conversation.append({'role': 'user', 'content': user_msg})  # add user turn
    reply = client.converse(conversation, system='You are a helpful AI assistant.')
    conversation.append({'role': 'assistant', 'content': reply}) # add assistant turn
    return reply

# Turn 1
r1 = chat_turn('What is a transformer?')
print(f'Turn 1:\n{r1}\n')

# Turn 2 — model remembers the previous exchange
r2 = chat_turn('What part of it is most novel?')
print(f'Turn 2:\n{r2}\n')

# Turn 3
r3 = chat_turn('Give a one-line Python example of the attention formula.')
print(f'Turn 3:\n{r3}')

## 6. Streaming Response

In [ ]:
# Streaming: tokens print as they are generated, not all at once
print('=== Streaming response ===')
full = client.stream(
    'Write a haiku about backpropagation.',
    max_tokens=100,
)
print(f'\nFull text ({len(full)} chars): {full!r}')

## 7. Utility: Pretty-print Settings

In [ ]:
def print_claude_config(mask_secrets: bool = True) -> None:
    """
    Pretty-print the loaded Claude configuration.

    Args:
        mask_secrets : if True, show only first 8 chars of auth tokens
    """
    s = read_claude_settings()                   # reload fresh from disk
    env = s.get('env', {})

    # Keys that contain secrets — mask these
    secret_keys = {'ANTHROPIC_AUTH_TOKEN', 'ANTHROPIC_API_KEY'}

    print('┌─────────────────────────────────────────────────────┐')
    print('│           Claude Code Settings                      │')
    print('├─────────────────────────────────────────────────────┤')
    print(f'│  model    : {s.get("model","(not set)"):<39}│')
    print('├─────────────────────────────────────────────────────┤')
    print('│  env vars:                                          │')
    for key, val in sorted(env.items()):
        if mask_secrets and key in secret_keys and val:
            display = val[:8] + '...'            # show only first 8 chars
        else:
            display = str(val) if val else '(empty)'
        # Truncate long values for display
        if len(display) > 38:
            display = display[:35] + '...'
        print(f'│  {key:<28}: {display:<20}│')
    print('└─────────────────────────────────────────────────────┘')

print_claude_config(mask_secrets=True)

## 8. Model Override Helper
Use a different model (haiku, opus) without changing the default.

In [ ]:
def get_model_id(alias: str, settings: Optional[Dict[str, Any]] = None) -> str:
    """
    Resolve a short model alias to the full model ID using modelOverrides.

    Args:
        alias    : short name, e.g. 'claude-haiku-4-5' or 'claude-sonnet-4-6'
        settings : pre-loaded settings dict (loaded if not provided)

    Returns:
        Full model ID string, e.g. 'global.anthropic.claude-haiku-4-5-20251001-v1:0'
        Falls back to alias itself if not found in overrides.
    """
    if settings is None:
        settings = read_claude_settings()

    overrides = settings.get('modelOverrides', {})  # the modelOverrides block
    return overrides.get(alias, alias)              # return override or alias as-is


# Show available models from settings
print('Available model aliases:')
for alias, model_id in settings.get('modelOverrides', {}).items():
    print(f'  {alias:<28} → {model_id}')

# Create a haiku client (cheaper / faster)
haiku_model = get_model_id('claude-haiku-4-5', settings)
haiku_client = ClaudeClient(model=haiku_model)
print(f'\nHaiku client model: {haiku_client.model}')

In [ ]:
# Quick question with the haiku model
reply = haiku_client.chat(
    prompt     = 'What is 2+2? Answer in one word.',
    max_tokens = 10,
)
print(f'Haiku reply: {reply}')

## 9. Summary — Helper Functions Reference

| Function / Class | Returns | Use |
|-----------------|---------|-----|
| `read_claude_settings()` | `dict` | Full parsed `~/.claude/settings.json` |
| `get_claude_env(key)` | `str\|None` | Single env var with OS fallback |
| `get_all_claude_env()` | `dict` | All env vars from settings |
| `get_model_id(alias)` | `str` | Resolve short alias → full model ID |
| `print_claude_config()` | `None` | Pretty-print config (secrets masked) |
| `ClaudeClient()` | client | Auto-configured Anthropic client |
| `client.chat(prompt)` | `str` | Single-turn chat |
| `client.converse(messages)` | `str` | Multi-turn with history |
| `client.stream(prompt)` | `str` | Streaming with live print |

### Reuse in other notebooks
```python
# At the top of any other notebook:
%run claude-1.ipynb          # or import if converted to .py

client = ClaudeClient()
reply  = client.chat('Your question here')
```